Import Libraries

In [1]:
import pandas as pd
import  matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from mlflow.data.pandas_dataset import PandasDataset
from imblearn.over_sampling import SMOTENC

import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
import boto3

s3_client = boto3.client(
    "s3",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    endpoint_url="http://minio.team-24.com:9000", # Use the API port
)

'''import os

os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio.team-24.com"'''

'import os\n\nos.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"\nos.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"\nos.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio.team-24.com"'

In [4]:
# Load the dataset from the CSV file sourced from Kaggle 
raw = pd.read_csv('../data/fraud_detection.csv')
# Drop the 'transaction_id' column from the raw dataframe, because it is not needed for analysis
raw = raw.drop('transaction_id', axis=1)
# Display the first 10 rows of the dataframe
raw.head(10)

,amount,merchant_type,device_type,label
0,46.93,travel,tablet,0
1,301.01,groceries,desktop,0
2,131.67,others,tablet,0
3,91.29,electronics,desktop,0
4,16.96,others,mobile,0
5,16.96,electronics,tablet,0
6,5.98,clothing,mobile,0
7,201.12,electronics,tablet,0
8,91.91,travel,mobile,0
9,123.13,groceries,tablet,0


In [ ]:
raw.info()

In [ ]:
raw['merchant_type'].value_counts()

In [ ]:
raw['device_type'].value_counts()

There are 3 indenpendent variables which are 
1. amount: Shows the amount of one transaction
2. merchant_type: The type of product/service a transaction is in
3. device_type: The type of product/service a transaction is in

Visualize each independent variable relation to the target variable

In [ ]:
# Boxplot: Amount vs Label
plt.figure(figsize=(8, 5))
sns.boxplot(x='label', y='amount', data=raw)
plt.title('Transaction Amount by Label')
plt.show()


Those which are fraudulent and not have similar median, mean, and quartiles. But those which are not fraudulent, have multiple transactions that tend to be higher in value. 

In [ ]:
# Countplot: Merchant Type vs Label
plt.figure(figsize=(10, 5))
sns.countplot(x='merchant_type', hue='label', data=raw)
plt.title('Merchant Type by Label')
plt.xticks(rotation=45)
plt.show()

There are no significant conclusion to be drawn from the relation between merchant_type and label.

In [ ]:
# Countplot: Device Type vs Label
plt.figure(figsize=(8, 5))
sns.countplot(x='device_type', hue='label', data=raw)
plt.title('Device Type by Label')
plt.xticks(rotation=45)
plt.show()

There are no significant conclusion to be drawn from the relation between merchant_type and label.


In [ ]:
# Grouped barplot to visualize merchant_type, device_type, and label in one graph
plt.figure(figsize=(12, 6))
sns.countplot(
    x='merchant_type',
    hue='label',
    data=raw,
    palette='Set2'
)
plt.title('Merchant Type by Label')
plt.xticks(rotation=45)
plt.legend(title='Label')
plt.tight_layout()
plt.show()

# FacetGrid to show device_type distribution within each merchant_type and label
g = sns.catplot(
    x='device_type',
    hue='label',
    col='merchant_type',
    data=raw,
    kind='count',
    col_wrap=3,
    height=4,
    palette='Set2'
)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Device Type and Label Distribution by Merchant')

In [ ]:
one_hot_data = pd.get_dummies(raw, columns=['merchant_type', 'device_type'], dtype='int')
# Display the first 10 rows of the one-hot encoded dataframe
one_hot_data.head(10)

In [ ]:
one_hot_data.corr()[['label']]

**Preprocessing Data**
Prepare data from raw dataframe and list the categorical variables for Sklearn Pipeline.

In [ ]:
categorical_features = ['merchant_type', 'device_type']
numerical_features = ['amount']
independent_var = categorical_features + numerical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    raw[independent_var], raw['label'], 
    test_size=0.2, random_state=42, stratify = raw['label'])

In [ ]:
sm = SMOTENC(categorical_features=categorical_features, random_state=42
             , sampling_strategy=0.4
             )

In [ ]:
X_res, y_res = sm.fit_resample(X_train, y_train)

In [ ]:
y_res.value_counts()

**Model: Hyperparameter Tuning**

Set Tracking to MLFlow uri

In [9]:
mlflow.set_tracking_uri("https://mlflow.team-24.com/")

In [ ]:
#%env MLFLOW_TRACKING_INSECURE_TLS=true
mlflow.set_experiment("fraud_detection_experiment")

In [ ]:
base_logreg = LogisticRegression(random_state=42, multi_class='ovr', penalty=None)
base_rf = RandomForestClassifier(random_state=42)
base_svc = SVC(random_state=42)
base_lgbm = LGBMClassifier(random_state=42)

skf = StratifiedKFold(shuffle = False, n_splits=5)

Create parameter grids for each model type

In [ ]:
param_logreg = {'model__solver': ['lbfgs','newton-cg','sag']}
param_rf = {'model__n_estimators': [50, 100, 250, 500]
            , 'model__min_samples_leaf': [1, 2, 5]
            , 'model__min_samples_split': [10, 25, 50, 100]
}
param_svc = {'model__kernel': ['linear', 'rbf', 'poly']}
param_grid = {'model__boosting_type': ['gbdt', 'dart', 'rf']
            , 'model__n_estimators': [50, 100, 250, 500]
            , 'model__num_leaves': [8, 32, 64, 128]
}

Create a pipeline for each type of Machine Learning algorithm.
Generally, logistic regression and SVM require normalization.
While tree based models are less sensitive to feature scales.

In [ ]:
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), numerical_features),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

preprocessor_rf = ColumnTransformer(
            [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
            remainder='passthrough'
) #OneHotEncoder is used to handle categorical features

preprocessor_lgbm = ColumnTransformer(
            [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
            remainder='passthrough'
) #OneHotEncoder is used to handle categorical features

pipeline_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', base_logreg)
])

pipeline_svc = Pipeline([
    ('preprocessor', preprocessor),
    ('model', base_svc)
])

pipeline_lgbm = Pipeline([
    ('preprocessor', preprocessor_lgbm),
    ('model', base_lgbm)
])

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor_rf),
    ('model', base_rf)
])

Start tracking model in MLFlow.

In [ ]:
#Define the data to be logged
train_df = pd.concat([X_train, y_train], axis=1)
train_dataset = PandasDataset(train_df, source="https://www.kaggle.com/datasets/sahideseker/fraud-detection-in-transactions-dataset")
trainsmote_df = pd.concat([X_train, y_train], axis=1)
trainsmote_dataset_v1 = PandasDataset(trainsmote_df, source="https://www.kaggle.com/datasets/sahideseker/fraud-detection-in-transactions-dataset")
test_df = pd.concat([X_test, y_test], axis=1)
test_dataset = PandasDataset(test_df, source="https://www.kaggle.com/datasets/sahideseker/fraud-detection-in-transactions-dataset")

In [ ]:
with mlflow.start_run(run_name="Logistic Regression Training") as run:
    run_id_logreg = run.info.run_id

    # Define hyperparameter tuning and cross-validation
    gs_logreg = GridSearchCV(
        pipeline_logreg, param_logreg
        , cv=skf, scoring="recall"
    )

    #Model fitting with cross-validation
    gs_logreg.fit(X_res, y_res)

    
    #log the the best parameters
    mlflow.log_param("logreg_bestparams", gs_logreg.best_params_)

    # Best model training
    gs_logreg.best_estimator_.fit(X_res, y_res)

    # log the metrics for best model
    y_pred_logreg = gs_logreg.best_estimator_.predict(X_test)
    
    # Logging the metrics
    # Log the metrics
    accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
    precision_logreg = precision_score(y_test, y_pred_logreg, average='weighted')
    recall_logreg = recall_score(y_test, y_pred_logreg, average='weighted')
    f1_logreg = f1_score(y_test, y_pred_logreg, average='weighted')
    metrics_logreg = {
        "accuracy": accuracy_logreg,
        "precision": precision_logreg,
        "recall": recall_logreg,
        "f1_score": f1_logreg
    }
    mlflow.log_metrics(metrics_logreg)

    #automatically determine the input and output schema
    signature_logreg = mlflow.models.infer_signature(X_res, gs_logreg.best_estimator_.predict(X_res))
        
    # Log the model
    mlflow.sklearn.log_model(sk_model = gs_logreg.best_estimator_
        , registered_model_name = "fraud-detection-model"
        , params = gs_logreg.best_params_
        , artifact_path="model"
        , signature = signature_logreg
        , tags={"model_type": "Logistic Regression",
        "class_balance": "60:40",
        "Description": "Logistic Regression model trained with SMOTE oversampling technique 60:40 class balance"
        }
    ) 